# Exercise 2 — Post-hoc XAI for Time-Series Forecasting

**PCS956 • Module C • Week 2**

## Aim

In this exercise you will construct a supervised forecasting problem from the **annual sunspot time series** provided by `statsmodels`.

You will analyse a Random Forest forecaster using:

- **TreeSHAP / TreeExplainer** — model-specific local feature attribution;
- **Permutation importance** — global model-agnostic predictive reliance;
- **Partial Dependence Plots (PDPs)** — global model-agnostic average model behaviour.

The main scientific issue is that **lagged predictors are temporally ordered and strongly dependent**. This makes careful interpretation of post-hoc explanations especially important.

> These methods explain the fitted forecasting model. They do not identify the physical mechanism that generates sunspot activity.

## Learning objectives

After completing the exercise, you should be able to:

- construct lagged predictors using only past observations;
- use a chronological train-test split to avoid future-to-past leakage;
- evaluate forecast performance before interpreting explanations;
- compute a local TreeSHAP explanation for one forecast;
- compute permutation importance on a held-out chronological test period;
- inspect PDPs for selected lag features;
- explain how dependence between neighbouring lags affects post-hoc XAI.

## 1. Load the sunspot time series

The `statsmodels` sunspots dataset contains annual measurements of sunspot activity.

### Task 1

Plot the complete time series and inspect its temporal structure.

In [ ]:
# Uncomment in Google Colab if needed:
# !pip -q install statsmodels shap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.datasets import sunspots

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

data = sunspots.load_pandas().data.copy()
data.columns = ["year", "sunspots"]

display(data.head())
print("Number of observations:", len(data))

plt.figure(figsize=(10, 4))
plt.plot(data["year"], data["sunspots"])
plt.xlabel("Year")
plt.ylabel("Sunspot activity")
plt.title("Annual sunspot activity")
plt.show()


### Discussion 1

1. Does the series show temporal dependence or recurring structure?
2. Why should observations from a time series not automatically be treated as independent and identically distributed?
3. Why is a random train-test split inappropriate for this forecasting exercise?

## 2. Construct lagged predictors

We predict the value at the current time using observations from previous time steps.

With 12 lagged predictors, each forecast uses the previous 12 annual observations as input features.

Every predictor for a target time is therefore available **before** that target.

### Task 2

Create 12 lag variables and remove rows that do not have a complete lag history.

In [ ]:
df = data.copy()

N_LAGS = 12

for lag in range(1, N_LAGS + 1):
    df[f"lag_{lag}"] = df["sunspots"].shift(lag)

df = df.dropna().reset_index(drop=True)

feature_cols = [
    f"lag_{lag}"
    for lag in range(1, N_LAGS + 1)
]

X = df[feature_cols]
y = df["sunspots"]

display(df.head())
print("Usable observations:", len(df))


## 3. Chronological train-test split

The final 20% of the time series is held out as a future test period.

This preserves temporal order and prevents training on observations that occur after the test targets.

### Task 3

Create the chronological split and report the training and test periods.

In [ ]:
split_idx = int(len(df) * 0.80)

X_train = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()

y_train = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()

year_train = df["year"].iloc[:split_idx]
year_test = df["year"].iloc[split_idx:]

print(
    "Training period:",
    int(year_train.iloc[0]),
    "-",
    int(year_train.iloc[-1])
)

print(
    "Test period:",
    int(year_test.iloc[0]),
    "-",
    int(year_test.iloc[-1])
)


## 4. Fit and evaluate a Random Forest forecaster

A Random Forest provides a nonlinear forecasting function and is compatible with both model-specific and model-agnostic XAI methods.

The test period is not used for training.

### Task 4

Fit the model, report **MAE** and **RMSE**, and compare observed and predicted values.

In [ ]:
model = RandomForestRegressor(
    n_estimators=400,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)

pred_test = model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    pred_test
)

rmse = mean_squared_error(
    y_test,
    pred_test
) ** 0.5

print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")

plt.figure(figsize=(10, 4))
plt.plot(
    year_test,
    y_test.to_numpy(),
    label="Observed"
)
plt.plot(
    year_test,
    pred_test,
    label="Predicted"
)
plt.xlabel("Year")
plt.ylabel("Sunspot activity")
plt.title("Forecast performance on the chronological test period")
plt.legend()
plt.show()


### Discussion 2

1. Does the model capture the broad temporal variation?
2. Where does the forecast appear to under- or over-predict?
3. Why should model performance be evaluated before explanation outputs are interpreted?
4. Why does reasonable forecast accuracy not guarantee that an explanation is faithful?

## 5. TreeSHAP — local attribution for one forecast

**SHAP (SHapley Additive exPlanations)** explains individual predictions using feature attributions based on Shapley values.

**TreeExplainer** is specialised for tree-based models and uses their internal structure to compute feature attributions efficiently.

In this exercise, TreeExplainer uses the **tree-path-dependent** formulation. The reference value and attributions are therefore determined using information represented by the fitted tree ensemble rather than an explicit external background dataset.

Here, the attribution values describe how individual lag features contribute to one forecast relative to the explainer's reference output.

### Task 5

Explain one forecast from the chronological test period and verify the additive reconstruction numerically.


In [ ]:
# Uncomment in Google Colab if needed:
# !pip -q install shap

import shap

explainer = shap.TreeExplainer(
    model,
    feature_perturbation="tree_path_dependent"
)

instance_idx = min(10, len(X_test) - 1)
x_one = X_test.iloc[[instance_idx]]

shap_values = explainer.shap_values(x_one)

values = np.asarray(shap_values).reshape(-1)

base_value = float(
    np.asarray(explainer.expected_value).reshape(-1)[0]
)

prediction = float(
    model.predict(x_one)[0]
)

reconstruction = base_value + values.sum()

contrib = pd.Series(
    values,
    index=feature_cols
).sort_values(
    key=np.abs,
    ascending=False
)

print("Explained year:", int(year_test.iloc[instance_idx]))
print("Observed value:", float(y_test.iloc[instance_idx]))
print("Model prediction:", prediction)
print("SHAP reference value:", base_value)
print("SHAP reconstruction:", reconstruction)
print(
    "Absolute reconstruction error:",
    abs(prediction - reconstruction)
)

display(
    contrib.to_frame("SHAP value")
)

contrib.sort_values().plot(
    kind="barh",
    figsize=(8, 5)
)
plt.xlabel("SHAP attribution")
plt.title("TreeSHAP attribution for one forecast")
plt.show()


### Discussion 3

1. Which lag has the largest absolute attribution for this forecast?
2. Which lags increase the forecast relative to the reference output?
3. Which lags decrease it?
4. Does a large SHAP attribution imply that the corresponding historical observation caused the future value?
5. Why can dependence and redundancy among neighbouring lags redistribute attribution?

## 6. Permutation importance — global predictive reliance

**Permutation importance** measures how much predictive performance changes when one feature is randomly shuffled while the fitted model remains unchanged.

### Task 6

Estimate permutation importance on the held-out chronological test period.

In [ ]:
result = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=30,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

perm = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": result.importances_mean,
    "importance_std": result.importances_std
}).sort_values(
    "importance_mean",
    ascending=False
)

display(perm)

plt.figure(figsize=(8, 5))
plt.barh(
    perm["feature"][::-1],
    perm["importance_mean"][::-1],
    xerr=perm["importance_std"][::-1]
)
plt.xlabel(
    "Increase in MAE after permutation "
    "(equivalent score decrease)"
)
plt.title("Permutation importance of lag features")
plt.show()


### Discussion 4

1. Which lag has the largest permutation importance?
2. Does the global ranking match the local TreeSHAP explanation?
3. Why is disagreement not necessarily an error?
4. Why can strongly correlated lag variables make single-feature permutation importance difficult to interpret?
5. What information remains available to the model when one lag is shuffled but neighbouring lags remain intact?

## 7. Partial Dependence — average model behaviour

**Partial Dependence** summarises how the fitted model's average prediction changes when one selected feature is set to different values while the remaining predictors retain their observed values in the supplied evaluation sample.

With strongly dependent lagged predictors, this can create feature combinations that are rare or unrealistic, so the curves must be interpreted cautiously.

### Task 7

Plot PDPs for `lag_1`, `lag_2`, and `lag_6`.


In [ ]:
fig, ax = plt.subplots(
    1,
    3,
    figsize=(12, 4)
)

PartialDependenceDisplay.from_estimator(
    model,
    X_test,
    features=["lag_1", "lag_2", "lag_6"],
    grid_resolution=25,
    method="brute",
    ax=ax
)

plt.suptitle("Partial dependence of selected lag features")
plt.tight_layout()
plt.show()


### Discussion 5

1. How does the fitted model's average prediction vary with `lag_1`?
2. Are the partial-dependence relationships monotonic?
3. Why can the marginal PDP construction be problematic when lag features are strongly dependent?
4. What additional diagnostic might you inspect before placing substantial weight on a PDP interpretation?

## 8. Compare the explanation methods

Complete the table.

| Method | Scope | Model dependence | Quantity characterised | Important limitation for lagged time series |
|---|---|---|---|---|
| TreeSHAP |  |  |  |  |
| Permutation importance |  |  |  |  |
| PDP |  |  |  |  |

### Final reflection

Write **5–7 sentences** addressing:

- which lagged observations the fitted model appears to rely on;
- whether the local and global explanation methods agree;
- how temporal dependence complicates interpretation;
- why the explanation outputs describe the **fitted forecaster**, not the physical process generating sunspot activity;
- one methodological change you would consider if the data exhibited a strong distribution shift over time.

## References

- Lundberg, S. M., & Lee, S.-I. (2017). *A Unified Approach to Interpreting Model Predictions.* Advances in Neural Information Processing Systems (NeurIPS), 30. https://arxiv.org/abs/1705.07874

- Lundberg, S. M., Erion, G., Chen, H., et al. (2020). *From Local Explanations to Global Understanding with Explainable AI for Trees.* Nature Machine Intelligence, 2, 56–67. https://doi.org/10.1038/s42256-019-0138-9

- Breiman, L. (2001). *Random Forests.* Machine Learning, 45, 5–32. https://doi.org/10.1023/A:1010933404324

- Friedman, J. H. (2001). *Greedy Function Approximation: A Gradient Boosting Machine.* The Annals of Statistics, 29(5), 1189–1232. https://doi.org/10.1214/aos/1013203451

- National Geophysical Data Center. *Yearly Sunspot Data, 1700–2008.* Distributed through the `statsmodels` datasets collection. https://www.statsmodels.org/stable/datasets/generated/sunspots.html

- Pedregosa, F., Varoquaux, G., Gramfort, A., et al. (2011). *Scikit-learn: Machine Learning in Python.* Journal of Machine Learning Research, 12, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html
